In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Build Driver Standings
-- MAGIC
-- MAGIC #### Sources
-- MAGIC 1. fact_session_results
-- MAGIC 1. dim_drivers
-- MAGIC
-- MAGIC #### Output Columns
-- MAGIC 1. season
-- MAGIC 1. driver id
-- MAGIC 1. driver name
-- MAGIC 1. nationality
-- MAGIC 1. race starts
-- MAGIC 1. total points
-- MAGIC 1. number of wins
-- MAGIC 1. number of podiums
-- MAGIC 1. standing position

-- COMMAND ----------

-- MAGIC %md
-- MAGIC
-- MAGIC #### Entity Relationship Diagram - Formula1 Gold Schema
-- MAGIC
-- MAGIC ![Formula1 Gold Data.png](../../z-course-images/formula1-gold-data-erd.png "Formula1 Gold Data.png")

In [0]:
%sql
select * from formula1_catalog.gold.dim_drivers

In [0]:
%sql
CREATE OR REPLACE VIEW formula1_catalog.gold.v_driver_standing
as 
(with driver_session_summary as 
(select
         fsr.season,
         dd.driver_id,
         dd.driver_name,
         dd.nationality_region,
         count(*) as race_starts,
         sum(fsr.points) as total_points,
         count_if(is_win) as number_of_wins,
         count_if(is_podium) as number_of_podiums
from formula1_catalog.gold.fact_session_results fsr
join formula1_catalog.gold.dim_drivers dd
on fsr.driver_id = dd.driver_id
group by fsr.season,
         dd.driver_id,
         dd.driver_name,
         dd.nationality_region)

select 
season,
driver_id,
driver_name,
nationality_region,
rank() over (partition by season order by total_points desc,number_of_wins desc) as standing_rank,
race_starts,
total_points,
number_of_wins,
number_of_podiums
from driver_session_summary);
select * from formula1_catalog.gold.v_driver_standing where season=2025;

         